# 01 — Simulate AI vs AI Games

Run **Hard (depth 9) vs Champion (depth 12)** games, record every move to SQLite.

**Testing mode**: `N_GAMES = 10` (5 per pairing) to validate the pipeline.  
**Full run**: `N_GAMES = 100_000` (50k per pairing).

In [1]:
import os, sys, time, json, sqlite3, random
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import datetime

# tqdm for progress bars (pip install tqdm if missing)
try:
    from tqdm.auto import tqdm
except ImportError:
    from tqdm import tqdm

# Ensure the engine package is importable
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from engine.board import initial_game_state, apply_move_to_board
from engine.ai import (
    get_next_best_move, get_all_possible_moves,
    apply_move_ai, is_game_over,
)
# play_one_game and worker live in an importable module so
# ProcessPoolExecutor can pickle them (notebook __main__ can't be pickled)
from engine.runner import play_one_game, worker as _worker

## Configuration

In [9]:
# === CONFIGURABLE PARAMETERS ===

# Number of games (total across both pairings)
N_GAMES = 500          # Testing mode — change to 100_000 for full run

# Search depths (3=Easy, 6=Medium, 9=Hard, 12=Champion)
DEPTH_A = 12           # "Hard"
DEPTH_B = 20           # "Champion"

# Move limit to prevent infinite games between equal-strength AIs
MAX_MOVES = 200

# If True, game move #1 is a uniformly random legal move.
# Useful for opener/response statistics so the start isn't deterministic.
RANDOM_FIRST_MOVE = True

# Parallelism
NUM_WORKERS = max(1, os.cpu_count() - 1)

# Database path
DB_PATH = os.path.join('data', 'games.sqlite')

# Per-side AI search configuration (passed to get_next_best_move)
# Keys supported by current AI engine:
#   max_ms, max_nodes, root_probe_nodes, stochastic_top_k
AI_SEARCH_A = {
    'max_ms': 5000,
    'max_nodes': None,
    'root_probe_nodes': 100,
    'stochastic_top_k': 3,
}

AI_SEARCH_B = {
    'max_ms': 5000,
    'max_nodes': None,
    'root_probe_nodes': 100,
    'stochastic_top_k': 3,
}

print(f"Config: {N_GAMES} games, depths {DEPTH_A} vs {DEPTH_B}")
print(f"Workers: {NUM_WORKERS}, Move limit: {MAX_MOVES}")
print(f"Random first move: {RANDOM_FIRST_MOVE}")
print(f"DB: {DB_PATH}")
print(f"AI A search config: {AI_SEARCH_A}")
print(f"AI B search config: {AI_SEARCH_B}")

Config: 500 games, depths 12 vs 20
Workers: 11, Move limit: 200
Random first move: True
DB: data/games.sqlite
AI A search config: {'max_ms': 5000, 'max_nodes': None, 'root_probe_nodes': 100, 'stochastic_top_k': 3}
AI B search config: {'max_ms': 5000, 'max_nodes': None, 'root_probe_nodes': 100, 'stochastic_top_k': 3}


## Engine Smoke Test

Run 1 game at low depth to verify everything works before the full run.

In [10]:
# play_one_game is imported from engine.runner (see imports cell above)

# --- Smoke test: 1 game at depth 3 ---
print("Running smoke test (depth 3 vs 3)...")
smoke_kwargs = {'max_ms': 10, 'max_nodes': None, 'root_probe_nodes': 50, 'stochastic_top_k': 3}
g, m = play_one_game(
    3,
    3,
    seed=42,
    white_ai_kwargs=smoke_kwargs,
    black_ai_kwargs=smoke_kwargs,
    random_first_move=RANDOM_FIRST_MOVE,
)
print(f"  Winner: {g['winner']}, Moves: {g['total_moves']}, "
      f"Termination: {g['termination']}, Time: {g['duration_ms']:.0f}ms")
print(f"  First 3 moves: {[(r['from_vertex'], r['to_vertex']) for r in m[:3]]}")
print("Smoke test passed.")

Running smoke test (depth 3 vs 3)...
  Winner: BLACK, Moves: 20, Termination: total_conversion, Time: 29ms
  First 3 moves: [('C1', 'C12'), ('C7', 'C6'), ('C2', 'C3')]
Smoke test passed.


## Database Setup

In [11]:
os.makedirs('data', exist_ok=True)

def init_db(db_path):
    """Create the games and moves tables if they don't exist."""
    conn = sqlite3.connect(db_path)
    c = conn.cursor()
    c.execute('''
        CREATE TABLE IF NOT EXISTS games (
            game_id       INTEGER PRIMARY KEY AUTOINCREMENT,
            white_depth   INTEGER NOT NULL,
            black_depth   INTEGER NOT NULL,
            winner        TEXT NOT NULL,
            total_moves   INTEGER NOT NULL,
            termination   TEXT NOT NULL,
            duration_ms   REAL NOT NULL,
            created_at    TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    c.execute('''
        CREATE TABLE IF NOT EXISTS moves (
            move_id       INTEGER PRIMARY KEY AUTOINCREMENT,
            game_id       INTEGER NOT NULL,
            move_number   INTEGER NOT NULL,
            color         TEXT NOT NULL,
            from_vertex   TEXT NOT NULL,
            to_vertex     TEXT NOT NULL,
            strikes       TEXT,
            upgrades      TEXT,
            board_after   TEXT,
            FOREIGN KEY (game_id) REFERENCES games(game_id)
        )
    ''')
    conn.commit()
    conn.close()
    print(f"Database initialised: {db_path}")


def insert_game(conn, game_record, move_records):
    """Insert one game and its moves into the database. Returns game_id."""
    c = conn.cursor()
    c.execute(
        '''INSERT INTO games (white_depth, black_depth, winner, total_moves, termination, duration_ms)
           VALUES (?, ?, ?, ?, ?, ?)''',
        (game_record['white_depth'], game_record['black_depth'],
         game_record['winner'], game_record['total_moves'],
         game_record['termination'], game_record['duration_ms'])
    )
    game_id = c.lastrowid
    
    rows = [
        (game_id, mr['move_number'], mr['color'], mr['from_vertex'],
         mr['to_vertex'], mr['strikes'], mr['upgrades'], mr['board_after'])
        for mr in move_records
    ]
    c.executemany(
        '''INSERT INTO moves (game_id, move_number, color, from_vertex, to_vertex, strikes, upgrades, board_after)
           VALUES (?, ?, ?, ?, ?, ?, ?, ?)''',
        rows
    )
    return game_id


init_db(DB_PATH)

Database initialised: data/games.sqlite


## Parallel Game Runner

In [12]:
# _worker is imported from engine.runner (pickleable by child processes)

def run_simulation(
    n_games,
    depth_a,
    depth_b,
    max_moves,
    num_workers,
    db_path,
    ai_a_kwargs=None,
    ai_b_kwargs=None,
    random_first_move=False,
):
    """Run n_games across two pairings and write results to SQLite.

    Pairing A: WHITE=depth_a, BLACK=depth_b (first half)
    Pairing B: WHITE=depth_b, BLACK=depth_a (second half)

    ai_a_kwargs are used whenever depth_a plays.
    ai_b_kwargs are used whenever depth_b plays.
    random_first_move randomises move #1 in every game.
    """
    ai_a_kwargs = dict(ai_a_kwargs or {})
    ai_b_kwargs = dict(ai_b_kwargs or {})

    half = n_games // 2
    tasks = []

    # Pairing A: depth_a (WHITE) vs depth_b (BLACK)
    for i in range(half):
        tasks.append((depth_a, depth_b, max_moves, i, ai_a_kwargs, ai_b_kwargs, random_first_move))

    # Pairing B: depth_b (WHITE) vs depth_a (BLACK)
    for i in range(n_games - half):
        tasks.append((depth_b, depth_a, max_moves, half + i, ai_b_kwargs, ai_a_kwargs, random_first_move))

    conn = sqlite3.connect(db_path)
    results = []
    batch = []
    batch_size = 100

    print(f"Starting {n_games} games with {num_workers} workers...")

    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        futures = {executor.submit(_worker, t): t for t in tasks}

        with tqdm(total=n_games, desc="Games", unit="game") as pbar:
            for future in as_completed(futures):
                game_rec, move_recs = future.result()
                batch.append((game_rec, move_recs))
                results.append(game_rec)
                pbar.update(1)

                # Batch insert every N games
                if len(batch) >= batch_size:
                    for gr, mr in batch:
                        insert_game(conn, gr, mr)
                    conn.commit()
                    batch = []

    # Flush remaining
    if batch:
        for gr, mr in batch:
            insert_game(conn, gr, mr)
        conn.commit()

    conn.close()
    return results


print(f"Ready to run {N_GAMES} games.")

Ready to run 500 games.


## Run Simulation

In [13]:
results = run_simulation(
    N_GAMES,
    DEPTH_A,
    DEPTH_B,
    MAX_MOVES,
    NUM_WORKERS,
    DB_PATH,
    ai_a_kwargs=AI_SEARCH_A,
    ai_b_kwargs=AI_SEARCH_B,
    random_first_move=RANDOM_FIRST_MOVE,
)
print(f"\nCompleted {len(results)} games.")

Starting 500 games with 11 workers...


Games:   0%|          | 0/500 [00:00<?, ?game/s]


Completed 500 games.


## Summary Statistics

In [14]:
import pandas as pd

df = pd.DataFrame(results)

print("=" * 50)
print("SIMULATION SUMMARY")
print("=" * 50)

print(f"\nTotal games: {len(df)}")
print(f"\nWin distribution:")
print(df['winner'].value_counts().to_string())

print(f"\nTermination reasons:")
print(df['termination'].value_counts().to_string())

print(f"\nAvg game length: {df['total_moves'].mean():.1f} moves")
print(f"Avg game duration: {df['duration_ms'].mean():.0f} ms")

print(f"\n--- Pairing A: Hard (WHITE) vs Champion (BLACK) ---")
pa = df[df['white_depth'] == DEPTH_A]
if len(pa) > 0:
    print(f"  Games: {len(pa)}")
    print(f"  WHITE wins: {(pa['winner'] == 'WHITE').sum()} ({(pa['winner'] == 'WHITE').mean()*100:.1f}%)")
    print(f"  BLACK wins: {(pa['winner'] == 'BLACK').sum()} ({(pa['winner'] == 'BLACK').mean()*100:.1f}%)")
    print(f"  Draws:      {(pa['winner'] == 'DRAW').sum()}")

print(f"\n--- Pairing B: Champion (WHITE) vs Hard (BLACK) ---")
pb = df[df['white_depth'] == DEPTH_B]
if len(pb) > 0:
    print(f"  Games: {len(pb)}")
    print(f"  WHITE wins: {(pb['winner'] == 'WHITE').sum()} ({(pb['winner'] == 'WHITE').mean()*100:.1f}%)")
    print(f"  BLACK wins: {(pb['winner'] == 'BLACK').sum()} ({(pb['winner'] == 'BLACK').mean()*100:.1f}%)")
    print(f"  Draws:      {(pb['winner'] == 'DRAW').sum()}")

SIMULATION SUMMARY

Total games: 500

Win distribution:
winner
BLACK    375
WHITE    125

Termination reasons:
termination
no_legal_moves      422
total_conversion     78

Avg game length: 30.8 moves
Avg game duration: 952 ms

--- Pairing A: Hard (WHITE) vs Champion (BLACK) ---
  Games: 250
  WHITE wins: 65 (26.0%)
  BLACK wins: 185 (74.0%)
  Draws:      0

--- Pairing B: Champion (WHITE) vs Hard (BLACK) ---
  Games: 250
  WHITE wins: 60 (24.0%)
  BLACK wins: 190 (76.0%)
  Draws:      0


## Verify Database

Quick sanity check that the SQLite file has the expected data.

In [8]:
conn = sqlite3.connect(DB_PATH)
game_count = pd.read_sql('SELECT COUNT(*) as n FROM games', conn).iloc[0]['n']
move_count = pd.read_sql('SELECT COUNT(*) as n FROM moves', conn).iloc[0]['n']
sample = pd.read_sql('SELECT * FROM moves WHERE game_id = 1 ORDER BY move_number LIMIT 5', conn)
conn.close()

print(f"Games in DB: {game_count}")
print(f"Moves in DB: {move_count}")
print(f"\nSample moves from game 1:")
sample

Games in DB: 1500
Moves in DB: 35391

Sample moves from game 1:


,move_id,game_id,move_number,color,from_vertex,to_vertex,strikes,upgrades,board_after
0,1,1,1,WHITE,C1,C12,[],[],"{""C2"": {""color"": ""WHITE"", ""isUpgraded"": false}..."
1,2,1,2,BLACK,C7,C6,[],[],"{""C2"": {""color"": ""WHITE"", ""isUpgraded"": false}..."
2,3,1,3,WHITE,C2,C3,[],[],"{""D1"": {""color"": ""WHITE"", ""isUpgraded"": false}..."
3,4,1,4,BLACK,C8,C9,[],[],"{""D1"": {""color"": ""WHITE"", ""isUpgraded"": false}..."
4,5,1,5,WHITE,D1,C1,[],[],"{""D2"": {""color"": ""WHITE"", ""isUpgraded"": false}..."
